In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import os
import h5py
import multiprocessing as mp
from joblib import Parallel, delayed
import re
from collections import defaultdict
from hdf5_creation import prepare_for_hdf5, update_hdf5

## Paths

In [ ]:
# path to retrieve mat files from
path_to_mat = r"C:\Users\andri\donders_internship_git_test\Human_SleepSCoring\DilonAndriesse\data\mat_files"
# path to save hdf5 file to
hdf5_path = r"C:\Users\andri\donders_internship_git_test\Human_SleepSCoring\DilonAndriesse\data\hdf5_files\test_h5.h5"

## Variables

In [ ]:
epoch_length = 10 #in seconds
pattern = re.compile(r'^(SC4\d+_\d|ST7\d+_\d|S\d+_\d)')

## Workflow

In [ ]:
files = np.ravel(os.listdir(path_to_mat))
subjects = defaultdict(list)
subject_fs = []

# get corresponding fs for subjects
# and create a dictionary with all files per subject
for file in files:
    match = pattern.match(file)
    if match:
        subject_name = match.group(1)
        if subject_name not in subjects:
            if 'SC' in subject_name or 'ST' in subject_name:
                fs = 100
            else:
                fs = 250
            subject_fs.append(fs)
        subjects[subject_name].append(file)

# iterate subjects and fs
for (key, recording), fs in zip(subjects.items(), subject_fs):
    with h5py.File(hdf5_path, 'a')  as database:
        # check if subject already exists in h5
        if key in list(database.keys()):
            print(f'{key} already exists, skipping...')
            continue
        else:
            # call script to create hdf5 file
            print(f'Starting on subject: {key}')
            print(f'sampling freq: {fs}')
            prepare_for_hdf5(key, recording, fs, path_to_mat, epoch_length, hdf5_path)